In [ ]:
import sys
sys.path.append('../')

In [ ]:
import pandas as pd
import random
from tqdm import tqdm
from os.path import join as pjoin
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
root = '../../data/filtered'

In [ ]:
xPath = pjoin(root, 'X.h5')
yPath = pjoin(root, 'Y.h5')

X = pd.read_hdf(xPath)
Y = pd.read_hdf(yPath)

In [ ]:
# from matplotlib import pyplot as plt
# import numpy as np
# from sklearn.cluster import KMeans

# num_clusters = 2
# kmeans = KMeans(n_clusters=num_clusters)
# kmeans.fit(well_data.GR.values.reshape(-1, 1))
# cluster_centers = kmeans.cluster_centers_
# cluster_centers = np.sort(cluster_centers, axis=0)
# shale_baseline = cluster_centers[0][0]
# sand_baseline = cluster_centers[1][0]

# # Print the estimated baselines
# print(f"Estimated Shale Baseline: {shale_baseline}")
# print(f"Estimated Sand Baseline: {sand_baseline}")

# def shale_volume(gamma_ray, gamma_ray_max, gamma_ray_min):
#     vshale = (gamma_ray - gamma_ray_min) / (gamma_ray_max - gamma_ray_min)
#     return round(vshale, 4)

# well_names = random.sample(list(X.UWI.unique()), 1)
# well = random.sample(list(well_names), 1)[0]
# well_data = X[X.UWI == well]
# well_y = Y[X.UWI == well]

# sand_base_line = well_data[:200]['GR'].min()
# shale_base_line = well_data[:200]['GR'].max()

# num_splits = 3
# split_len = len(well_data['GR'])//num_splits


# start, end = 0, split_len
# for i in range(1, num_splits + 1):
#     gr_zone = well_data['GR'][start:end]
#     well_data.loc[gr_zone.index, 'VSH_CALC'] = shale_volume(gr_zone, gr_zone.quantile(0.99), gr_zone.quantile(0.01))
#     start = end
#     if i == num_splits - 1:
#         end = len(well_data['GR'])
#     else:
#         end += split_len
# vsh_calc = shale_volume(well_data.GR, well_data.GR.quantile(0.99), 40)
# print(mean_absolute_error(well_data.VSH, well_data.VSH_CALC), mean_absolute_error(well_data.VSH, vsh_calc))
# print(r2_score(well_data.VSH, well_data.VSH_CALC), r2_score(well_data.VSH, vsh_calc))
# plt.figure(figsize=(20, 5))
# vsh_calc.plot(label='VSH_CALC')
# well_data.VSH_CALC.plot(label='VSH_CALC_ZONED', linestyle='--', color='r')
# well_data.VSH.plot(label='VSH', color='r')
# plt.legend()

# vsh_calc = shale_volume(well_data.GR, 97, 45)
# print(mean_absolute_error(well_data.VSH, well_data.VSH_CALC), mean_absolute_error(well_data.VSH, vsh_calc))
# print(r2_score(well_data.VSH, well_data.VSH_CALC), r2_score(well_data.VSH, vsh_calc))
# plt.figure(figsize=(20, 5))
# vsh_calc.plot(label='VSH_CALC')
# well_data.VSH_CALC.plot(label='VSH_CALC_ZONED', linestyle='--', color='r')
# well_data.VSH.plot(label='VSH', color='r')
# plt.legend()

In [ ]:
Y = pd.concat([Y, X.PHI, X.SW], axis = 1)
X = X.drop(['PHI', 'SW'], axis = 1)

In [ ]:
root = '../data/processed'

In [ ]:
blind_well = random.sample(list(X.UWI.unique()), 10)

In [ ]:
blind_X = X[X.UWI.isin(blind_well) == True]
blind_Y = Y[X.UWI.isin(blind_well) == True]

Y = Y[X.UWI.isin(blind_well) == False]
X = X[X.UWI.isin(blind_well) == False]

In [ ]:
blind_X.to_hdf(pjoin(root, 'blind_X.h5'), key='X', mode='w')
blind_Y.to_hdf(pjoin(root, 'blind_Y.h5'), key='Y', mode='w')

X.to_hdf(pjoin(root, 'X.h5'), key='X', mode='w')
Y.to_hdf(pjoin(root, 'Y.h5'), key='Y', mode='w')

In [ ]:
import numpy as np
from tqdm import tqdm
import patchify as pf
from sklearn.preprocessing import StandardScaler

def create_patches(X, Y, well_names, data_config, num_features):
    # get parameters of patch
    patch_size = data_config['patch']['patch_size']
    stride = data_config['patch']['stride']
    well_name_column = data_config['well_name_column']
    x_patches, y_patches = [], []

    # iterate over wells
    for well in tqdm(well_names, desc='Creating Patches'):

        # get data of each well
        well_x = X[X[well_name_column] == well]
        well_y = Y.loc[well_x.index]

        # drop well name column
        well_x = well_x.drop([well_name_column], axis=1)

        # create patches if well has data more than patch size
        if well_x.shape[0] >= patch_size:

            # create patches
            well_x_patches = pf.patchify(well_x.values, (patch_size, num_features), step=stride)
            well_x_patches = well_x_patches.squeeze()

            well_y_patches = pf.patchify(well_y.values, (patch_size, well_y.values.shape[-1]), step=stride)
            well_y_patches = well_y_patches.squeeze()

            if well_x_patches.ndim == 3:
                for i, j in zip(well_x_patches, well_y_patches):
                    x_patches.append(i)
                    y_patches.append(j)
                    
            #if well has only one patch
            else:
                x_patches.append(well_x_patches)
                y_patches.append(well_y_patches)

    return np.array(x_patches), np.array(y_patches)

import numpy as np
import pandas as pd
from os.path import join as pjoin

import torch

from sklearn.model_selection import train_test_split

def prepare_data(config, blind=False):
    print("Preparing the data...")

    # Load the parameters from the config file
    data_config = config['data']
    root = config['root']

    data_config['x_file_name'] = data_config['x_file_name'] if not config['model']['use_lora'] else 'lora_' + data_config['x_file_name']
    data_config['y_file_name'] = data_config['y_file_name'] if not config['model']['use_lora'] else 'lora_' + data_config['y_file_name']

    # Load the path to the processed data
    x_path = pjoin(root, data_config['processed_data_path'], data_config['x_file_name'])
    y_path = pjoin(root, data_config['processed_data_path'], data_config['y_file_name'])

    # Load the data from hdf5 files
    X = pd.read_hdf(x_path)
    Y = pd.read_hdf(y_path)

    # Get the unique well names
    well_names = X.UWI.unique()

    # Drop the columns that are not needed in the training
    X.drop(data_config['drop_columns'], axis=1, inplace=True)

    # Scale the data
    X = scale_data(X, data_config)

    # check if the data is patch based, if yes, create patches else model will be trained on the point data
    if data_config['patch_based']:

        # Create patches from the data
        num_features = X.shape[1] - 1
        x_patches, y_patches = create_patches(X, Y, well_names, data_config, num_features)

        # Get the number of classes
        num_classes = len(np.unique(y_patches[:, :, 0]))

        if not blind:
            # Split the data into train and validation
            (
                x_train, 
                x_val, 
                y_train, 
                y_val
            ) = train_test_split(
                x_patches, 
                y_patches, 
                test_size=data_config['split_size'], 
                random_state=config['random_state']
            )

            # Convert the data to PyTorch tensors
            x_train = torch.tensor(x_train).float()
            y_train = torch.tensor(y_train)
            x_val = torch.tensor(x_val).float()
            y_val = torch.tensor(y_val)
        else:
            x_train = torch.tensor(x_patches).float()
            y_train = torch.tensor(y_patches)
            x_val = None
            y_val = None
    else:
        # drop the well name column
        X = X.drop([data_config['well_name_column']], 
                   axis=1)
        
        # Get the number of classes
        num_classes = len(np.unique(Y))

        # Split the data into train and validation
        (
            x_train, 
            x_val, 
            y_train, 
            y_val
        ) = train_test_split(
            X, 
            Y, 
            test_size=data_config['split_size'], 
            random_state=config['random_state']
        )

    print(f"Number of classes: {num_classes} and shape of x_train: {x_train.shape}")
    return x_train, x_val, y_train, y_val, num_classes

def scale_data(X, data_config):
    if 'lat' in data_config['scaled_columns']:
        lat_min, lat_max = X.lat.min(), X.lat.max()
        X.lat = (X.lat - lat_min) / (lat_max - lat_min)

    if 'lng' in data_config['scaled_columns']:
        lng_min, lng_max = X.lng.min(), X.lng.max()
        X.lng = (X.lng - lng_min) / (lng_max - lng_min)

    if 'DEPT' in data_config['scaled_columns']:
        depth_min, depth_max = X.DEPT.min(), X.DEPT.max()
        X.DEPT = (X.DEPT - depth_min) / (depth_max - depth_min)

    if 'ILD' in data_config['scaled_columns']:
        scaler_ild = StandardScaler()
        X.ILD = scaler_ild.fit_transform(X.ILD.values.reshape(-1, 1))

    if 'GR' in data_config['scaled_columns']:
        scaler_gr = StandardScaler()
        X.GR = scaler_gr.fit_transform(X.GR.values.reshape(-1, 1))

    if 'NPHI' in data_config['scaled_columns']:
        scaler_nphi = StandardScaler()
        X.NPHI = scaler_nphi.fit_transform(X.NPHI.values.reshape(-1, 1))

    if 'DPHI' in data_config['scaled_columns']:
        scaler_dphi = StandardScaler()
        X.DPHI = scaler_dphi.fit_transform(X.DPHI.values.reshape(-1, 1))

    if 'W_Tar' in data_config['scaled_columns']:
        scaler_w_tar = StandardScaler()
        X.W_Tar = scaler_w_tar.fit_transform(X.W_Tar.values.reshape(-1, 1))

    if 'SW' in data_config['scaled_columns']:
        scaler_sw = StandardScaler()
        X.SW = scaler_sw.fit_transform(X.SW.values.reshape(-1, 1))

    if 'VSH' in data_config['scaled_columns']:
        scaler_vsh = StandardScaler()
        X.VSH = scaler_vsh.fit_transform(X.VSH.values.reshape(-1, 1))

    if 'VSH_CALC' in data_config['scaled_columns']:
        scaler_vsh_calc = StandardScaler()
        X.VSH_CALC = scaler_vsh_calc.fit_transform(X.VSH_CALC.values.reshape(-1, 1))

    if 'PHI' in data_config['scaled_columns']:
        scaler_phi = StandardScaler()
        X.PHI = scaler_phi.fit_transform(X.PHI.values.reshape(-1, 1))

    return X

In [ ]:
config = {}
config['data'] = {}
config['model'] = {}
config['data']['patch'] = {}

config['model']['use_lora'] = False

config['data']['x_file_name'] = 'X.h5'
config['data']['y_file_name'] = 'Y.h5'
config['data']['processed_data_path'] = 'data/multitask_processed'
config['data']['drop_columns'] = ['W_Tar','lat','lng','DEPT','RW']
config['data']['scaled_columns'] = ['GR','NPHI','DPHI','ILD','VSH']
config['data']['patch_based'] = True
config['data']['patch']['patch_size'] = 150
config['data']['patch']['stride'] = 150
config['data']['well_name_column'] = 'UWI'
config['data']['split_size'] = 0.2
config['data']['lithology_classes'] = {'Sand': 0,
                                       'ShalySand': 1,
                                       'SandyShale': 2,
                                       'Shale': 3,
                                       'Coal': 4,
                                       'CementedSand': 5}

config['root'] = '..'
config['random_state'] = 42

batch_size = 64

In [ ]:
x_train, x_val, y_train, y_val, num_classes = prepare_data(config)

In [ ]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [ ]:
train_dataset = TensorDataset(x_train, y_train)
trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(x_val, y_val)
valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
regression_criterion = torch.nn.MSELoss()
classification_criterion = torch.nn.CrossEntropyLoss()

In [ ]:
from model.multi_task_vit import build_model

In [ ]:
config['data']['num_features'] = x_train.shape[-1]
config['model']['activation'] = 'relu'
config['model']['dim'] = 150
config['model']['depth'] = 1
config['model']['heads'] = 4
config['model']['mlp_dim'] = 256
config['model']['channels'] = 1
config['model']['dim_head'] = 128

config['trainer'] = {}
config['trainer']['device'] = 'cuda'

In [ ]:
model = build_model(config)
model = model.to(config['trainer']['device'])

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
import numpy as np
from tqdm import tqdm

import torch
from einops import rearrange
import torch.nn as nn

from sklearn.metrics import confusion_matrix

from utils.misc import update_best_metrices

def train_engine(
        epoch, model, 
        train_loader, 
        regression_criterion,
        classification_criterion, 
        optimizer, 
        num_epochs, 
        loss_weights,
        device
    ):

    train_loss = 0.0
    sw_train_loss = 0.0
    lith_train_loss = 0.0
    phi_train_loss = 0.0
    train_correct = 0

    gt, pred = [], []

    model.train()
    for batch_inputs, batch_labels in tqdm(train_loader, 
                                           total=len(train_loader), 
                                           desc=f"Train - Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()

        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device).to(torch.float32)

        # Forward pass
        lith_output, phi_output, sw_output = model(batch_inputs)

        lith_batch_labels = batch_labels[:, :, 0]
        phi_output_labels = batch_labels[:, :, 1:2]
        sw_output_labels = batch_labels[:, :, 2:]

        lith_batch_labels = lith_batch_labels.long()

        outputs_ = rearrange(lith_output, 'b n d -> b d n')
        lith_loss = classification_criterion(outputs_, lith_batch_labels)

        sw_loss = regression_criterion(sw_output, sw_output_labels)
        phi_loss = regression_criterion(phi_output, phi_output_labels)

        loss = (lith_loss*loss_weights[0]) + (phi_loss*loss_weights[1]) + (sw_loss*loss_weights[2])
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Calculate training accuracy
        predicted = torch.argmax(nn.Softmax(dim = -1)(lith_output), dim=-1)
        train_correct += (((lith_batch_labels == predicted).sum(-1).float().mean().item())/batch_inputs.shape[1])*100

        gt.append(lith_batch_labels.cpu())
        pred.append(predicted.cpu())

        train_loss += loss.item()
        sw_train_loss += sw_loss.item()
        phi_train_loss += phi_loss.item()
        lith_train_loss += lith_loss.item()

    cm = confusion_matrix(torch.cat(gt, dim=0).view(-1), torch.cat(pred, dim=0).view(-1))

    # Calculate average training loss and accuracy for the epoch
    train_loss /= len(train_loader)
    sw_train_loss /= len(train_loader)
    lith_train_loss /= len(train_loader)
    phi_train_loss /= len(train_loader)
    train_accuracy = train_correct / len(train_loader)

    return train_loss, lith_train_loss, phi_train_loss, sw_train_loss, train_accuracy, cm

def validation_engine(
        epoch, 
        model, 
        val_loader, 
        regression_criterion,
        classification_criterion,
        num_epochs, 
        loss_weights,
        device
    ):
    
    val_loss = 0.0
    sw_val_loss = 0.0
    lith_val_loss = 0.0
    phi_val_loss = 0.0
    val_correct = 0

    gt_val, pred_val = [], []
    
    # Evaluate on the validation set
    model.eval()
    for batch_inputs_val, batch_labels_val in tqdm(val_loader, 
                                                   total=len(val_loader), 
                                                   desc=f"Val - Epoch {epoch+1}/{num_epochs}"):
        
        batch_inputs_val = batch_inputs_val.to(device)
        batch_labels_val = batch_labels_val.to(device)

        with torch.no_grad():
            lith_output_val, phi_output_val, sw_output_val = model(batch_inputs_val)


        lith_batch_labels_val = batch_labels_val[:, :, 0]
        phi_batch_labels_val = batch_labels_val[:, :, 1:2]
        sw_batch_labels_val = batch_labels_val[:, :, 2:]

        lith_batch_labels_val = lith_batch_labels_val.long()
        val_outputs_ = rearrange(lith_output_val, 'b n d -> b d n')
        lith_loss_val = classification_criterion(val_outputs_, lith_batch_labels_val)
        
        sw_loss_val = regression_criterion(sw_output_val, sw_batch_labels_val)
        phi_loss_val = regression_criterion(phi_output_val, phi_batch_labels_val)

        sw_std_loss_val = regression_criterion(sw_output_val.squeeze().std(dim=1), sw_batch_labels_val.squeeze().std(dim=1))
        sw_mean_loss_val = regression_criterion(sw_output_val.squeeze().mean(dim=1), sw_batch_labels_val.squeeze().mean(dim=1))

        loss_val = (lith_loss_val*loss_weights[0]) + (phi_loss_val*loss_weights[1]) + (sw_loss_val*loss_weights[2]) + (sw_std_loss_val*loss_weights[2])# + (sw_mean_loss_val*loss_weights[2])


        val_predicted = torch.argmax(nn.Softmax(dim = -1)(lith_output_val), dim=-1)
        val_correct += (((val_predicted == lith_batch_labels_val).sum(-1).float().mean().item())/batch_inputs_val.shape[1])*100

        val_loss += loss_val.item()
        sw_val_loss += sw_loss_val.item()
        lith_val_loss += lith_loss_val.item()
        phi_val_loss += phi_loss_val.item()

        gt_val.append(lith_batch_labels_val.cpu())
        pred_val.append(val_predicted.cpu())

    cm_val = confusion_matrix(torch.cat(gt_val, dim=0).view(-1), torch.cat(pred_val, dim=0).view(-1))

    val_loss /= len(val_loader)
    sw_val_loss /= len(val_loader)
    lith_val_loss /= len(val_loader)
    phi_val_loss /= len(val_loader)
    val_accuracy = val_correct / len(val_loader)

    return val_loss, lith_val_loss, phi_val_loss, sw_val_loss, val_accuracy, cm_val

In [ ]:
config['model']['loss_weights'] = [1, 1, 20]

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
train_sw_losses = []
val_sw_losses = []
train_lith_losses = []
val_lith_losses = []
train_phi_losses = []
val_phi_losses = []
best_loss = np.inf
best_cm_val = None
best_cm = None
best_model_chkpt = None
best_optim_chkpt = None
best_epoch = 1
best_accuracy = 0.0

for epoch in range(200):
    train_loss, lith_train_loss, phi_train_loss, sw_train_loss, train_accuracy, cm = train_engine(epoch, 
                                                model, 
                                                trainloader, 
                                                regression_criterion,
                                                classification_criterion, 
                                                optimizer, 
                                                200,
                                                config['model']['loss_weights'],
                                                'cuda')
    
    val_loss, lith_val_loss, phi_val_loss, sw_val_loss, val_accuracy, cm_val = validation_engine(epoch, 
                                                model, 
                                                valloader, 
                                                regression_criterion,
                                                classification_criterion, 
                                                200,
                                                config['model']['loss_weights'],
                                                'cuda')

    train_losses.append(train_loss)
    train_sw_losses.append(sw_train_loss)
    train_lith_losses.append(lith_train_loss)
    train_phi_losses.append(phi_train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_sw_losses.append(sw_val_loss)
    val_lith_losses.append(lith_val_loss)
    val_phi_losses.append(phi_val_loss)
    val_accuracies.append(val_accuracy)

    # Print the progress for the current epoch
    print(f"Epoch {epoch+1}/{200}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, SW Loss Train: {sw_train_loss:.4f}, Lith Loss Train: {lith_train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%, SW Loss Val: {sw_val_loss:.4f}, Lith Loss Val: {lith_val_loss:.4f}")
    
    (
        best_loss, 
        best_accuracy, 
        best_epoch, 
        best_cm, 
        best_cm_val, 
        best_model_chkpt, 
        best_optim_chkpt
    ) = update_best_metrices(
        val_loss, 
        val_accuracy, 
        epoch, 
        cm, 
        cm_val, 
        model, 
        optimizer, 
        best_loss, 
        best_accuracy, 
        best_epoch, 
        best_cm, 
        best_cm_val, 
        best_model_chkpt, 
        best_optim_chkpt
    )

    if epoch - (best_epoch) > 10:
        print("Early stopping")
        break

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
_, ax = plt.subplots(1, 5, figsize=(20, 5))
ax[0].plot(train_losses, label='train')
ax[0].plot(val_losses, label='val')
ax[0].set_title('Total Loss')
ax[0].legend()

ax[1].plot(train_lith_losses, label='train')
ax[1].plot(val_lith_losses, label='val')
ax[1].set_title('Lith Loss')
ax[1].legend()

ax[2].plot(train_sw_losses, label='train')
ax[2].plot(val_sw_losses, label='val')
ax[2].set_title('SW Loss')
ax[2].legend()

ax[3].plot(train_phi_losses, label='train')
ax[3].plot(val_phi_losses, label='val')
ax[3].set_title('PHI Loss')
ax[3].legend()

ax[4].plot(train_accuracies, label='train')
ax[4].plot(val_accuracies, label='val')
ax[4].set_title('Accuracy')
ax[4].legend()

plt.show()

In [ ]:
config['data']['x_file_name'] = 'blind_X.h5'
config['data']['y_file_name'] = 'blind_Y.h5'
x_train, _, y_train, _, num_classes = prepare_data(config)

blind_dataset = TensorDataset(x_train, y_train)
blind_loader = DataLoader(blind_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
mae_criterion = torch.nn.L1Loss()

In [ ]:
blind_loader = valloader

In [ ]:
model.eval()
val_correct = 0
phi_loss = 0
sw_loss = 0
count = 1
for inp, gt in blind_loader:
    cls,phi,sw = model(inp.to('cuda'))
    predicted = torch.argmax(nn.Softmax(dim = -1)(cls), dim=-1)

    val_correct += (((predicted == gt[:, :, 0].cuda()).sum(-1).float().mean().item())/inp.shape[1])*100
    phi_loss += mae_criterion(phi.squeeze(), gt[:, :, 1].cuda()).item()
    sw_loss += mae_criterion(sw.squeeze(), gt[:, :, 2].cuda()).item()
    print(val_correct/count, phi_loss/count, sw_loss/count)
    count += 1
    print('ok')

In [ ]:
# check = {}
# for i, (inp, gt) in enumerate(blind_loader):
#     temp = {}
#     cls,phi,sw = model(inp.to('cuda'))
#     for j, (sw_pred, sw_gt, phi_pred, phi_gt) in enumerate(zip(sw.squeeze(), gt[:, :, 2].cuda(), phi.squeeze(), gt[:, :, 1].cuda())):
#         if mae_criterion(sw_pred, sw_gt)<0.02:
#             temp[j] = mae_criterion(sw_pred, sw_gt)
#             plt.plot(sw_pred.cpu().detach().numpy(), label='pred')
#             plt.plot(sw_gt.cpu().detach().numpy(), label='gt')
#             plt.legend()
#             plt.ylim(0, 1.1)
#             plt.show()

#             plt.plot(phi_pred.cpu().detach().numpy(), label='pred')
#             plt.plot(phi_gt.cpu().detach().numpy(), label='gt')
#             plt.legend()
#             plt.show()

#             print("="*100)
#     check[i] = temp

In [ ]:
_, ax = plt.subplots(1, 2, figsize=(20, 5))

ax[0].scatter(gt[:, :, 1].reshape(-1), phi[:, :].cpu().detach().numpy().reshape(-1))
ax[0].plot([gt[:, :, 1].min().item(), gt[:, :, 1].max().item()], [gt[:, :, 1].min().item(), gt[:, :, 1].max().item()], color='red')
ax[0].set_title('PHI')

ax[1].scatter(gt[:, :, 2].reshape(-1), sw[:, :].cpu().detach().numpy().reshape(-1))
ax[1].plot([gt[:, :, 2].min().item(), gt[:, :, 2].max().item()], [gt[:, :, 2].min().item(), gt[:, :, 2].max().item()], color='red')
ax[1].set_title('SW')

plt.show()

In [ ]:
rand_idx = np.random.randint(gt.shape[0])
_, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].plot(gt[rand_idx, :, 0].cpu().numpy(), label='gt')
ax[0].plot(predicted[rand_idx, :].cpu().detach().numpy(), label='pred')
ax[0].set_title('Lithology')
ax[0].legend()

ax[1].plot(gt[rand_idx, :, 1].cpu().numpy(), label='gt')
ax[1].plot(phi[rand_idx, :].cpu().detach().numpy(), label='pred')
ax[1].set_title(f'PHI MAE: {mae_criterion(phi[rand_idx, :], gt[rand_idx, :, 1].cuda()).item()}')
ax[1].legend()

ax[2].plot(gt[rand_idx, :, 2].cpu().numpy(), label='gt')
ax[2].plot(sw[rand_idx, :].cpu().detach().numpy(), label='pred')
ax[2].set_title(f'SW MAE: {mae_criterion(sw[rand_idx, :], gt[rand_idx, :, 2].cuda()).item()}')
ax[2].legend()
ax[2].set_ylim(0, 1)

plt.show()

print(gt[rand_idx, :, 2].std(), sw[rand_idx, :].std())

In [ ]:
plt.plot(phi[rand_idx, :].cpu().detach().numpy())
plt.plot(gt[rand_idx, :, 1])

In [ ]:
(phi[rand_idx, :].cpu().detach() - gt[rand_idx, :, 1]).abs().mean()